# 01 — Colab Training (VOC → COCO)

Edge Vision Model: from-scratch nano NMS-free detector.
This notebook mounts Drive, installs deps, clones the repo, and runs the full training ladder:
**overfit-20 sanity → VOC full → COCO**. Checkpoints and logs save to Drive.

Runtime: **GPU (T4)** recommended.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Edge_Vision_Model'  # artifacts root on Drive
import os; os.makedirs(DRIVE, exist_ok=True)


In [ ]:
!nvidia-smi -L || echo 'no GPU - switch runtime type'
!python -V


## 1. Install deps + clone repo (code imports from the repo, not from Drive)

In [ ]:
REPO_URL = 'https://github.com/avneeshjadhav04/edge-vision-model'
BRANCH = 'main'
!pip -q install albumentations pyyaml onnx onnxruntime onnxsim onnxscript
%cd /content
!rm -rf edge-vision-model
!git clone -b $BRANCH $REPO_URL
%cd edge-vision-model
import sys; sys.path.insert(0, '/content/edge-vision-model')


## 2. Download Pascal VOC

In [ ]:
from data.download import download_voc
download_voc(root='/content/datasets/VOC')


## 3. Overfit sanity (20 images, target mAP >= 0.90)

Fast correctness gate: ~5-10 min on T4. If this fails, do **not** start full training.


In [ ]:
!python -m scripts.overfit_test --root /content/datasets/VOC --epochs 300 \
    --batch-size 8 --img-size 320 --device cuda --save-dir /content/runs/overfit


In [ ]:
# copy overfit artifacts to Drive
!cp -r /content/runs/overfit "$DRIVE"/ 2>/dev/null || true


## 4. Full VOC training (07+12 trainval -> VOC2007 test)

Target: **70-75 mAP@0.5** on VOC2007 test. Expect **<1h** on T4 at 640px/120 epochs.


In [ ]:
!python -m scripts.train --dataset voc --root /content/datasets/VOC \
    --epochs 120 --batch-size 32 --img-size 640 --device cuda \
    --save-dir /content/runs/voc 2>&1 | tail -40


In [ ]:
!cp -r /content/runs/voc "$DRIVE"/ 2>/dev/null || true


## 5. (Optional) quick eval of best VOC weights

In [ ]:
!python -m scripts.eval --dataset voc --root /content/datasets/VOC \
    --weights /content/runs/voc/best.pt --img-size 640 2>&1 | tail -25


## 6. COCO training

From VOC init (backbone/neck transfer; head reinit happens automatically for 80 classes
via strict=False load) or from scratch - set `INIT_FROM` accordingly.
Budget on T4: ~2.5-3s/it at bs=64 -> ~100 epochs about 3 days. For the 35+ mAP goal use
the full 300-epoch schedule if you have the quota; 100 epochs typically lands ~30-35.


In [ ]:
INIT_FROM = '/content/runs/voc/best.pt'   # or '' for from-scratch
EPOCHS = 100
cmd = ('python -m scripts.train --dataset coco --root /content/datasets/coco '
       f'--epochs {EPOCHS} --batch-size 64 --img-size 640 --device cuda '
       '--save-dir /content/runs/coco')
if INIT_FROM:
    cmd += ' --init-from ' + INIT_FROM
print(cmd)


In [ ]:
from data.download import download_coco
download_coco(root='/content/datasets/coco', splits=('val2017',), with_train=True)


In [ ]:
import subprocess
with open('/content/coco_train.log', 'w') as f:
    p = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
print('exit', p.returncode)
!tail -30 /content/coco_train.log


In [ ]:
!cp -r /content/runs/coco "$DRIVE"/ 2>/dev/null || true
!cp /content/coco_train.log "$DRIVE"/ 2>/dev/null || true


## 7. Training curves

In [ ]:
import json, matplotlib.pyplot as plt
def curve(log_json, title):
    h = json.load(open(log_json))
    ep = [x['epoch'] for x in h]
    loss = [x['loss'] for x in h]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(ep, loss); ax[0].set_title(f'{title} loss'); ax[0].set_xlabel('epoch')
    m = [(x['epoch'], x['mAP']) for x in h if 'mAP' in x]
    if m:
        ax[1].plot(*zip(*m)); ax[1].set_title(f'{title} mAP'); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.savefig(f'{title.lower()}_curve.png', dpi=150)
    plt.show()
try: curve('/content/runs/voc/log.json', 'VOC')
except FileNotFoundError: print('VOC log missing')
try: curve('/content/runs/coco/log.json', 'COCO')
except FileNotFoundError: print('COCO log missing')


## 8. Final numbers (fill README from these)

In [ ]:
!python -m scripts.flops
!python -m scripts.eval --dataset coco --root /content/datasets/coco \
    --weights /content/runs/coco/best.pt --img-size 640 2>&1 | tail -3
